## If working in Google Colab, run this first
Run the cell below at the start of every session. It downloads the course files to your Google Drive and keeps them up to date.
When you see "Setup complete", you are ready to run the rest of the notebook.

**Do not edit or save your work inside the `AIforMovementScience` folder.**
Use **File > Save a copy in Drive** and work on your own copy.

In [ ]:
# [course-setup] Run this cell first, every session (or use Runtime > Run all)
import os, sys
try:
    from google.colab import drive
    os.chdir('/content')                  # start from a valid directory
    drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/AIforMovementScience'
    if os.path.isdir(REPO + '/.git'):
        # already downloaded before: just update it
        !git -C "$REPO" fetch origin main && git -C "$REPO" reset --hard origin/main
    else:
        # first time, or a broken/partial copy: download a fresh copy
        !rm -rf "$REPO"
        !git clone https://github.com/dguari1/AIforMovementScience.git "$REPO"
except ImportError:
    # not in Colab: walk up from here until the repository root is found
    REPO = os.getcwd()
    while REPO != '/' and not os.path.isdir(os.path.join(REPO, 'lectures', 'data')):
        REPO = os.path.dirname(REPO)

WORKDIR = REPO + '/lectures'
os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)
print('Setup complete. You can now run the rest of the notebook.')

# Convolutional Neural Networks for Images

In module 05 we applied 1D kernels to signals. The same idea works for **images**: the kernel is now a small **matrix** that slides over the image. At each position, the kernel values are multiplied by the pixel values under the kernel, and the products are added. The result is one pixel of the output image.

<div style="text-align:center">
<img src="images/Convolution_arithmetic_-_Full_padding_no_strides.gif" alt="plot" width="400" />
</div>

- Blue $\rightarrow$ input image
- Gray $\rightarrow$ kernel
- Green $\rightarrow$ output image

## Learning goals
- see how different kernels change an image (blur, edges, sharpen), and design your own kernel
- understand the shape of the output of a 2D convolution layer (number of kernels, padding, stride)
- train a small CNN on handwritten digits (MNIST), and look at what each layer does
- reuse a trained CNN for a new task (**transfer learning**: digits → clothes)

In notebook 06.03 we apply these ideas to images of human postures.

In [ ]:
import os
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
import copy
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from scipy.signal import convolve2d
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(0); np.random.seed(0)

## 1. How kernels change an image

We use a grayscale version of the image of 06.01 (one channel, values from 0 to 255).

In [ ]:
img = np.asarray(Image.open('data/sample_image.jpg').convert('L'), dtype=float)   # 'L' = grayscale
print('Image shape:', img.shape)
plt.figure(figsize=(6, 4)); plt.imshow(img, cmap='gray'); plt.axis('off'); plt.title('Input image'); plt.show()

Here are some classical kernels:

| Kernel | What it does |
| --- | --- |
| **mean 3 × 3** | each pixel becomes the mean of its 3 × 3 neighborhood: a small blur |
| **mean 9 × 9** | the mean of a larger neighborhood: a stronger blur |
| **vertical edges** | large output where the image changes from left to right (vertical lines) |
| **horizontal edges** | large output where the image changes from top to bottom (horizontal lines) |
| **sharpen** | the center pixel minus its neighbors: makes details clearer |
| **all edges (Laplacian)** | responds to changes in all directions |

The function `show_kernel` applies a kernel with `convolve2d` (from SciPy), and shows the input, the kernel (with its values), and the output.

In [ ]:
kernels = {
    'mean 3x3':         np.ones((3, 3)) / 9,
    'mean 9x9':         np.ones((9, 9)) / 81,
    'vertical edges':   np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]),     # Sobel kernel
    'horizontal edges': np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]]),     # Sobel kernel
    'sharpen':          np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]]),
    'all edges':        np.array([[-1, -1, -1], [-1, 8, -1], [-1, -1, -1]]),
}

def show_kernel(image, kernel, name=''):
    out = convolve2d(image, kernel, mode='same', boundary='symm')
    fig, ax = plt.subplots(1, 3, figsize=(14, 3.5), gridspec_kw={'width_ratios': [3, 1.2, 3]})
    ax[0].imshow(image, cmap='gray'); ax[0].set_title('Input'); ax[0].axis('off')
    ax[1].imshow(kernel, cmap='coolwarm', vmin=-np.abs(kernel).max(), vmax=np.abs(kernel).max())
    if kernel.shape[0] <= 5:
        for (r, c), v in np.ndenumerate(kernel):
            ax[1].text(c, r, f'{v:.2g}', ha='center', va='center', fontsize=10)
    ax[1].set_title(f'Kernel: {name}'); ax[1].set_xticks([]); ax[1].set_yticks([])
    if kernel.sum() == 0:                     # edge kernels: output is positive and negative, show the size of the change
        ax[2].imshow(np.abs(out), cmap='gray', vmax=np.percentile(np.abs(out), 99))
    else:
        ax[2].imshow(np.clip(out, 0, 255), cmap='gray', vmin=0, vmax=255)
    ax[2].set_title('Output'); ax[2].axis('off')
    plt.show()

for name, k in kernels.items():
    show_kernel(img, k, name)

Things to notice:
- The **mean** kernels have positive values that add up to 1: the output is a blurred image with the same brightness. The larger kernel blurs more.
- The **edge** kernels have values that add up to 0: in a region with the same color the output is 0 (black), and only the **changes** remain. The vertical-edges kernel finds vertical lines; the horizontal-edges kernel finds horizontal lines.
- The **sharpen** kernel adds up to 1 (so the brightness stays the same), but it makes the differences with the neighbors larger.

(As in 05.02a, the mathematical convolution flips the kernel; CNN layers do not. For symmetric kernels this is not important.)

### Activity: design your own kernel

Change the values of `my_kernel` and run the cell. Ideas:
- a kernel that finds **diagonal** edges
- a 1 × 9 kernel (one row) that blurs only in the horizontal direction
- what happens if the values add up to 2? Or to 0.5?

In [ ]:
my_kernel = np.array([[ 0,  1,  2],
                      [-1,  0,  1],
                      [-2, -1,  0]])
show_kernel(img, my_kernel, 'my kernel')

## 2. Convolution layers in PyTorch

In a CNN, the kernels are **not** chosen by us: they start with random values and are **learned** from the data with gradient descent (module 05). The basic operations of a CNN are:
1. convolution (many kernels)
2. a nonlinearity (for example ReLU)
3. pooling (downsampling)

<div style="text-align:center">
<img src="images/2d-convolution.jpg" alt="plot" width="800" />
</div>

The 2D convolution layer of PyTorch is

```python
torch.nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0)
```

- `in_channels`: the number of channels of the input (1 for grayscale, 3 for color)
- `out_channels`: the number of kernels, so the number of output images (**feature maps**)
- `kernel_size`: the size of each kernel (3 means 3 × 3)
- `padding`: the number of pixels added around the image
- `stride`: how many pixels the kernel moves each time

The input must have the shape **(batch, channels, height, width)**. Let us see how the settings change the output shape.

In [ ]:
x = torch.randn(1, 1, 20, 20)          # one grayscale image of 20 x 20 pixels
for settings in [dict(kernel_size=3), dict(kernel_size=3, padding=1), dict(kernel_size=5),
                 dict(kernel_size=3, padding=1, stride=2)]:
    conv = nn.Conv2d(1, 6, **settings)
    print(f'Conv2d(1, 6, {settings}): input {tuple(x.shape)} -> output {tuple(conv(x).shape)} | parameters: {sum(p.numel() for p in conv.parameters())}')

- Without padding, a 3 × 3 kernel loses 1 pixel on each side (20 → 18); a 5 × 5 kernel loses 2 (20 → 16).
- With `padding=1`, a 3 × 3 kernel keeps the size.
- With `stride=2`, the output is 2 times smaller.
- 6 kernels of 3 × 3 have 6 × 9 = 54 weights, plus 6 biases = 60 parameters.

A `Conv2d` layer does exactly what `convolve2d` did in part 1. To show this, we put the vertical-edges kernel into a `Conv2d` layer by hand, and apply it to our image. We also apply a layer with **random** kernels (as at the start of training).

In [ ]:
x = torch.tensor(img, dtype=torch.float32)[None, None]          # shape (1, 1, H, W)

edge_layer = nn.Conv2d(1, 1, kernel_size=3, padding=1, bias=False)
with torch.no_grad():
    edge_layer.weight[:] = torch.tensor(kernels['vertical edges'][::-1, ::-1].copy(), dtype=torch.float32)   # flipped: Conv2d does not flip
random_layer = nn.Conv2d(1, 4, kernel_size=3, padding=1)          # 4 random kernels

with torch.no_grad():
    edge_out = edge_layer(x)[0, 0].numpy()
    random_out = random_layer(x)[0].numpy()

fig, ax = plt.subplots(1, 5, figsize=(18, 3))
ax[0].imshow(np.abs(edge_out), cmap='gray', vmax=np.percentile(np.abs(edge_out), 99)); ax[0].set_title('Conv2d with the\nvertical-edges kernel')
for k in range(4):
    ax[k + 1].imshow(random_out[k], cmap='gray'); ax[k + 1].set_title(f'random kernel {k}')
for a in ax: a.axis('off')
plt.show()

The first image is the same as the vertical-edges output of part 1. Random kernels give random mixtures of blur and edges. Training changes them into kernels that are useful for the task.

## 3. A CNN for handwritten digits (MNIST)

MNIST is a classic dataset of handwritten digits (0 to 9), 28 × 28 pixels, grayscale. We use a subset saved in the repository: 5,000 training images and 2,000 test images (`data/mnist`).

In [ ]:
d = np.load('data/mnist/mnist_subset.npz')
Xd_train, yd_train, Xd_test, yd_test = d['X_train'], d['y_train'], d['X_test'], d['y_test']
print('Train:', Xd_train.shape, '| Test:', Xd_test.shape, '| pixel values:', Xd_train.min(), 'to', Xd_train.max())

fig, ax = plt.subplots(1, 10, figsize=(15, 2))
for k in range(10):
    ax[k].imshow(Xd_train[k], cmap='gray'); ax[k].set_title(int(yd_train[k])); ax[k].axis('off')
plt.show()

We scale the pixels to the range 0–1, add the channel dimension, and make the data loaders. To keep the training fast, we use **1,000** training images.

In [ ]:
def make_loader(X, y, shuffle):
    X = torch.tensor(X, dtype=torch.float32).unsqueeze(1) / 255     # (N, 1, 28, 28), values 0-1
    return DataLoader(TensorDataset(X, torch.tensor(y, dtype=torch.long)), batch_size=64, shuffle=shuffle)

digits_train = make_loader(Xd_train[:1000], yd_train[:1000], shuffle=True)
digits_test = make_loader(Xd_test, yd_test, shuffle=False)

The CNN has two convolution stages and a small fully connected network:

| Layer | Output shape |
| --- | --- |
| input | 1 × 28 × 28 |
| `conv1`: 5 kernels 3 × 3, then max pooling and ReLU | 5 × 13 × 13 |
| `conv2`: 10 kernels 5 × 5, then max pooling and ReLU | 10 × 4 × 4 |
| flatten: the **embedding** | 160 |
| `fc1` (ReLU) → `fc2` | 100 → 10 class scores |

The model also keeps the feature maps of each stage, so we can look at them later.

In [ ]:
class DigitCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 5, kernel_size=3)
        self.conv2 = nn.Conv2d(5, 10, kernel_size=5)
        self.pool = nn.MaxPool2d(2)
        self.relu = nn.ReLU()
        self.fc1 = nn.Linear(160, 100)
        self.fc2 = nn.Linear(100, 10)

    def forward(self, x):
        self.maps1 = self.relu(self.pool(self.conv1(x)))         # stage 1
        self.maps2 = self.relu(self.pool(self.conv2(self.maps1)))  # stage 2
        self.embedding = self.maps2.flatten(1)                   # 160 numbers
        return self.fc2(self.relu(self.fc1(self.embedding)))

def run_training(model, train_loader, test_loader, epochs, lr=0.01, verbose=True):
    """Train with Adam and cross-entropy; return the train and test accuracy after each epoch."""
    optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    criterion = nn.CrossEntropyLoss()
    hist = {'train': [], 'test': []}
    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            optimizer.zero_grad(); loss = criterion(model(xb), yb); loss.backward(); optimizer.step()
        hist['train'].append(accuracy(model, train_loader)); hist['test'].append(accuracy(model, test_loader))
        if verbose and (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1:2d} | train accuracy {hist['train'][-1]:.3f} | test accuracy {hist['test'][-1]:.3f}")
    return hist

@torch.no_grad()
def accuracy(model, loader):
    model.eval()
    return np.mean(np.concatenate([(model(xb).argmax(1) == yb).numpy() for xb, yb in loader]))

torch.manual_seed(0)
digit_model = DigitCNN()
print(f'Parameters: {sum(p.numel() for p in digit_model.parameters()):,}')
hist = run_training(digit_model, digits_train, digits_test, epochs=20)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(range(1, 21), hist['train'], marker='o', label='Train')
plt.plot(range(1, 21), hist['test'], marker='o', label='Test')
plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.ylim(0.5, 1.01); plt.legend(); plt.grid(alpha=0.3)
plt.title('MNIST, 1,000 training images')
plt.show()

With only 1,000 training images and about 18,000 parameters, the small CNN recognizes most of the test digits. The train accuracy reaches 1.0 and the test accuracy is lower: the model overfits a little. (Here we only *look* at the test curve; to *choose* the number of epochs we would use a validation set, as in 05.02c.)

### What does the network do?

Let us look at the 5 kernels of `conv1` after training, and at the feature maps of a few test digits.

In [ ]:
w1 = digit_model.conv1.weight.detach()[:, 0].numpy()            # (5, 3, 3)
fig, ax = plt.subplots(1, 5, figsize=(12, 2.5))
for k in range(5):
    ax[k].imshow(w1[k], cmap='coolwarm', vmin=-np.abs(w1).max(), vmax=np.abs(w1).max())
    ax[k].set_title(f'conv1 kernel {k}'); ax[k].set_xticks([]); ax[k].set_yticks([])
plt.show()

def show_maps(model, x, title):
    model.eval()
    with torch.no_grad():
        model(x[None])
    fig, ax = plt.subplots(3, 10, figsize=(16, 5))
    for a in ax.ravel(): a.axis('off')
    ax[0, 0].imshow(x[0], cmap='gray'); ax[0, 0].set_title(title)
    for k in range(5):
        ax[1, k].imshow(model.maps1[0, k], cmap='gray'); ax[1, k].set_title(f'stage 1, map {k}', fontsize=8)
    for k in range(10):
        ax[2, k].imshow(model.maps2[0, k], cmap='gray'); ax[2, k].set_title(f'stage 2, map {k}', fontsize=8)
    ax[0, 2].imshow(model.embedding[0][None], cmap='gray', aspect='auto'); ax[0, 2].set_title('embedding (160 numbers)')
    ax[0, 2].set_position([0.3, 0.72, 0.6, 0.08])
    plt.show()

X_test_t = torch.tensor(Xd_test, dtype=torch.float32).unsqueeze(1) / 255
for i in [0, 1, 2]:
    show_maps(digit_model, X_test_t[i], f'digit {yd_test[i]}')

- **Stage 1** (13 × 13): the maps still look like the digit; each kernel highlights different edges or strokes.
- **Stage 2** (4 × 4): the maps are small and abstract; they show *where* some combinations of strokes are.
- The **embedding** (160 numbers) is the input of the fully connected layers. Different digits give different patterns of numbers.
- Some maps are completely black for every digit: after the ReLU their values are always 0, so these kernels are not used ("dead" kernels). This is common in small networks.

## 4. Transfer learning: from digits to clothes

**Fashion-MNIST** has the same format as MNIST (28 × 28 grayscale, 10 classes), but the images show clothes and shoes. We have a subset in `data/mnist` too.

In [ ]:
f = np.load('data/mnist/fashion_mnist_subset.npz')
Xf_train, yf_train, Xf_test, yf_test = f['X_train'], f['y_train'], f['X_test'], f['y_test']
fashion_labels = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

fig, ax = plt.subplots(1, 10, figsize=(16, 2))
for k in range(10):
    i = np.flatnonzero(yf_train == k)[0]
    ax[k].imshow(Xf_train[i], cmap='gray'); ax[k].set_title(fashion_labels[k], fontsize=9); ax[k].axis('off')
plt.show()

fashion_train = make_loader(Xf_train[:1000], yf_train[:1000], shuffle=True)
fashion_test = make_loader(Xf_test, yf_test, shuffle=False)

**(a) Use the digit model directly.** What happens if we give clothes to the model trained on digits? It still gives one of 10 outputs, but its outputs mean "digit 0 to 9", not "T-shirt, trouser, ...".

In [ ]:
print(f'Digit model on Fashion-MNIST: test accuracy = {accuracy(digit_model, fashion_test):.3f}  (chance = 0.100)')

**(b) Transfer learning.** A CNN that learned to recognize digits has learned general features of small images: edges, strokes, corners. These features can also be useful for clothes. So we:
1. copy the trained digit model,
2. **freeze** the convolution layers (`requires_grad_(False)`): their kernels do not change,
3. train **only** the fully connected layers with 1,000 Fashion-MNIST images.

**(c) From scratch.** For comparison, we train the same CNN from random weights with the same 1,000 images.

In [ ]:
transfer_model = copy.deepcopy(digit_model)
transfer_model.conv1.requires_grad_(False)          # freeze the convolution layers
transfer_model.conv2.requires_grad_(False)
frozen = sum(p.numel() for p in transfer_model.parameters() if not p.requires_grad)
print(f'Transfer learning: {frozen:,} convolution parameters are frozen (from the digit model); the fully connected layers are trained')
hist_transfer = run_training(transfer_model, fashion_train, fashion_test, epochs=20, verbose=False)

torch.manual_seed(0)
scratch_model = DigitCNN()
hist_scratch = run_training(scratch_model, fashion_train, fashion_test, epochs=20, verbose=False)

print(f"Fashion-MNIST test accuracy | transfer learning: {hist_transfer['test'][-1]:.3f} | from scratch: {hist_scratch['test'][-1]:.3f}")
plt.figure(figsize=(7, 4))
plt.plot(range(1, 21), hist_transfer['test'], marker='o', label='Transfer (frozen digit kernels)')
plt.plot(range(1, 21), hist_scratch['test'], marker='o', label='From scratch')
plt.xlabel('Epoch'); plt.ylabel('Test accuracy'); plt.legend(); plt.grid(alpha=0.3)
plt.title('Fashion-MNIST, 1,000 training images')
plt.show()

The model with frozen digit kernels reaches a good accuracy (far above the 0.10 of chance), although its convolution kernels **never saw a piece of clothing**. So the features learned for digits (edges, strokes) are also useful for clothes: they **transfer**.

However, here transfer learning is **not better** than training from scratch. We also tested it with 100 and 300 training images, and from clothes to digits: training from scratch was always as good or better. The reasons:
- the network is very small: its convolution layers have only about 1,300 parameters, so they are easy to learn again from a few hundred images;
- MNIST is a narrow dataset (centered digits, black background), so its features are not very general.

Transfer learning becomes really important when:
- the new dataset is **small**, and
- the pretrained model was trained on a **very large and varied** dataset (for example ImageNet, with 1.2 million images).

This is the situation in notebook 06.03: we have only about 1,000 images of postures, so we will start from a CNN pretrained on ImageNet.

## Summary
- A kernel is a small matrix; different kernels blur, sharpen or find edges.
- A CNN learns its kernels from the data. The first layers find simple patterns (edges, strokes); deeper layers combine them.
- `Conv2d` settings (number of kernels, kernel size, padding, stride) set the shape of the output.
- A trained CNN can be reused for a new task (transfer learning), often with only the last layers trained.

## Activity
1. Train `DigitCNN` on MNIST with 100, 1,000 and 5,000 training images. How does the test accuracy change?
2. In part 4, freeze only `conv1` (train `conv2` and the fully connected layers). Is the result better than freezing both?